# Embedding Layer


In [1]:
from pathlib import Path

import torch
import torch.nn as nn

torch.manual_seed(0)

LECTURE_NOTE_TITLE = 'Embedding Layer'

vocab = ["<pad>", "I", "like", "LLMs", "."]
token_to_id = {token: idx for idx, token in enumerate(vocab)}
batch_ids = torch.tensor([
    [token_to_id["I"], token_to_id["like"], token_to_id["LLMs"], token_to_id["."]],
    [token_to_id["I"], token_to_id["LLMs"], token_to_id["<pad>"], token_to_id["<pad>"]],
])

print(f'Lecture note: {LECTURE_NOTE_TITLE}')
print(batch_ids)


Lecture note: Embedding Layer
tensor([[1, 2, 3, 4],
        [1, 3, 0, 0]])


## Demo A: one-hot vectors and embedding lookup are the same linear map


In [2]:
E = torch.tensor(
    [
        [0.0, 0.0, 0.0],
        [1.0, 0.5, -0.5],
        [0.2, 1.2, 0.3],
        [1.5, -0.2, 0.7],
        [-0.3, 0.4, 1.1],
    ]
)
token_id = token_to_id["LLMs"]
one_hot = torch.nn.functional.one_hot(torch.tensor(token_id), num_classes=len(vocab)).float()

print('one_hot @ E:', one_hot @ E)
print('row lookup   :', E[token_id])
print('equal        :', torch.allclose(one_hot @ E, E[token_id]))


one_hot @ E: tensor([ 1.5000, -0.2000,  0.7000])
row lookup   : tensor([ 1.5000, -0.2000,  0.7000])
equal        : True


## Demo B: sequence lookup and output shape


In [3]:
embedding = nn.Embedding(num_embeddings=len(vocab), embedding_dim=4)
looked_up = embedding(batch_ids)

print('input shape :', tuple(batch_ids.shape))
print('output shape:', tuple(looked_up.shape))
print('token row for "I":', embedding.weight[token_to_id['I']])
print('first vector in batch:', looked_up[0, 0])
print('same row selected:', torch.allclose(looked_up[0, 0], embedding.weight[token_to_id['I']]))


input shape : (2, 4)
output shape: (2, 4, 4)
token row for "I": tensor([ 0.5988, -1.5551, -0.3414,  1.8530], grad_fn=<SelectBackward0>)
first vector in batch: tensor([ 0.5988, -1.5551, -0.3414,  1.8530], grad_fn=<SelectBackward0>)
same row selected: True


## Demo C: padding_idx gets no gradient


In [4]:
padded_embedding = nn.Embedding(num_embeddings=len(vocab), embedding_dim=4, padding_idx=token_to_id['<pad>'])
loss = padded_embedding(batch_ids).sum()
loss.backward()

print('grad for <pad> row:')
print(padded_embedding.weight.grad[token_to_id['<pad>']])
print('grad for "I" row:')
print(padded_embedding.weight.grad[token_to_id['I']])


grad for <pad> row:
tensor([0., 0., 0., 0.])
grad for "I" row:
tensor([2., 2., 2., 2.])


## Demo D: vocabulary size controls parameter count


In [5]:
for vocab_size in [1_000, 10_000, 50_000]:
    for d_model in [128, 768]:
        params = vocab_size * d_model
        print(f'V={vocab_size:>6}, d_model={d_model:>3} -> {params:>10,} parameters')
    print('-' * 60)


V=  1000, d_model=128 ->    128,000 parameters
V=  1000, d_model=768 ->    768,000 parameters
------------------------------------------------------------
V= 10000, d_model=128 ->  1,280,000 parameters
V= 10000, d_model=768 ->  7,680,000 parameters
------------------------------------------------------------
V= 50000, d_model=128 ->  6,400,000 parameters
V= 50000, d_model=768 -> 38,400,000 parameters
------------------------------------------------------------


## Demo E: weight tying


In [6]:
class TinyUntiedLM(nn.Module):
    def __init__(self, vocab_size: int, d_model: int):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        hidden = self.embed(ids).mean(dim=1)
        return self.lm_head(hidden)


class TinyTiedLM(nn.Module):
    def __init__(self, vocab_size: int, d_model: int):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.embed.weight

    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        hidden = self.embed(ids).mean(dim=1)
        return self.lm_head(hidden)


untied = TinyUntiedLM(vocab_size=50_000, d_model=768)
tied = TinyTiedLM(vocab_size=50_000, d_model=768)

print('untied params:', sum(p.numel() for p in untied.parameters()))
print('tied params  :', sum(p.numel() for p in tied.parameters()))

hidden = tied.embed(batch_ids).mean(dim=1)
logits_from_head = tied.lm_head(hidden)
logits_from_dot_products = hidden @ tied.embed.weight.T
print('tied head equals hidden @ embedding.T:', torch.allclose(logits_from_head, logits_from_dot_products))


untied params: 76800000
tied params  : 38400000
tied head equals hidden @ embedding.T: True


## Demo F: static token rows, contextual hidden states


In [7]:
bank_vocab = {'<pad>': 0, 'river': 1, 'bank': 2, 'money': 3}
bank_embedding = nn.Embedding(len(bank_vocab), 3)

river_context = torch.tensor([[bank_vocab['river'], bank_vocab['bank']]])
money_context = torch.tensor([[bank_vocab['money'], bank_vocab['bank']]])

static_bank = bank_embedding.weight[bank_vocab['bank']]
contextual_river = bank_embedding(river_context)[0, 1] + bank_embedding(river_context).mean(dim=1)[0]
contextual_money = bank_embedding(money_context)[0, 1] + bank_embedding(money_context).mean(dim=1)[0]

print('same static row every time:', static_bank)
print('bank after river context   :', contextual_river)
print('bank after money context   :', contextual_money)


same static row every time: tensor([-0.3309, -1.1809, -0.9358], grad_fn=<SelectBackward0>)
bank after river context   : tensor([-0.7182, -2.2515, -1.9491], grad_fn=<AddBackward0>)
bank after money context   : tensor([-0.4575, -2.2479, -1.6809], grad_fn=<AddBackward0>)


## Rasbt and nanochat


In [8]:
references = {
    'rasbt': {
        'files': [
            'https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/ch02.ipynb',
            'https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/03_bonus_embedding-vs-matmul/embeddings-and-linear-layers.ipynb',
        ],
        'core': 'Teach the one-hot equals embedding-row identity very explicitly, then use nn.Embedding for the real implementation.',
    },
    'nanochat': {
        'files': [
            'https://github.com/karpathy/nanochat/blob/master/nanochat/gpt.py',
        ],
        'core': 'Token embeddings live inside the GPT model and feed the rest of the network; production code treats them as one block inside a larger system.',
    },
}

for name, info in references.items():
    print(name.upper())
    print('core :', info['core'])
    print('files:')
    for file in info['files']:
        print(' -', file)
    print()


RASBT
core : Teach the one-hot equals embedding-row identity very explicitly, then use nn.Embedding for the real implementation.
files:
 - https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/ch02.ipynb
 - https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/03_bonus_embedding-vs-matmul/embeddings-and-linear-layers.ipynb

NANOCHAT
core : Token embeddings live inside the GPT model and feed the rest of the network; production code treats them as one block inside a larger system.
files:
 - https://github.com/karpathy/nanochat/blob/master/nanochat/gpt.py

